# Population Realizations and Spectra

This notebook starts from the same semi-analytic model, computes the expected number of binaries in the PTA band, and then generates discrete realizations of the stochastic background.

## Imports and Model Setup

We reuse the same parameter set as in the semi-analytic notebook, but now focus on Monte Carlo realizations and frequency-bin spectra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp

import fastropop
from fastropop import SemiAnalyticPopulation


In [ ]:
population_params = {
    "n0": 10**-90.4153,
    "alphaM": -1.3800,
    "Mstar": 10**8.8272 * fastropop.MsunMKS,
    "betaz": -0.1711,
    "z0": 4.70,
}

sampling_grids = {
    "Mgrid": jnp.geomspace(fastropop.Mmin / fastropop.kg, fastropop.Mmax / fastropop.kg, 300),
    "zgrid": jnp.linspace(fastropop.zmin, fastropop.zmax, 300),
    "fgrid": jnp.geomspace(fastropop.fminNG15 * fastropop.s, fastropop.fmaxNG15 * fastropop.s, 300),
}

pop = SemiAnalyticPopulation(
    population_params=population_params,
    sampling_grids=sampling_grids,
)


## Expected Number of Binaries and Reference Spectrum

The quantity `Nbinaries_mean` is the ensemble-average number of binaries in the chosen mass, redshift, and frequency range. We also compute a smooth reference `h_c(f)` curve for comparison.

In [ ]:
Nbinaries_mean = pop.compute_Nbinaries(verbose=True)
freqs = 10 ** np.arange(-9.0, -7.4, 0.2)
hc2_values = np.array([np.sqrt(pop.hc2(float(f))) for f in freqs])

Nbinaries_mean


## One Poisson Realization

A single realization first draws the total number of binaries from a Poisson distribution and then samples the binary parameters from the model. The resulting PTA-style spectrum is one possible realization of the background.

In [ ]:
Nbinaries_draw = int(pop.generate_poisson_realization(1, Nbinaries_mean=Nbinaries_mean, key=0)[0])
distM, distz, distlog10f = pop.sample_dist(Nbinaries=Nbinaries_draw, key=1)
spec = fastropop.binning(distM, distz, distlog10f, freqs=freqs, hc2_values=hc2_values, do_plot=True)

Nbinaries_draw, spec.shape


## Many Realizations

To estimate the scatter induced by Poisson fluctuations and discrete source sampling, we generate several realizations and compare them to the ensemble reference spectrum.

In [ ]:
tabreal, log10f, median, q_low, q_high = pop.compute_many_realizations(
    Nbinaries_mean=Nbinaries_mean,
    nrealizations=20,
    freqs=freqs,
    hc2_values=hc2_values,
    key=2,
    do_plot=True,
)

tabreal.shape
